In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/heavy-equipment-selling-price-prediction-challenge/sample_submission.csv
/kaggle/input/competitions/heavy-equipment-selling-price-prediction-challenge/train.csv
/kaggle/input/competitions/heavy-equipment-selling-price-prediction-challenge/metadata.csv
/kaggle/input/competitions/heavy-equipment-selling-price-prediction-challenge/test.csv


In [ ]:
train = pd.read_csv(
    "/kaggle/input/competitions/heavy-equipment-selling-price-prediction-challenge/train.csv"
)

test = pd.read_csv(
    "/kaggle/input/competitions/heavy-equipment-selling-price-prediction-challenge/test.csv"
)

submission = pd.read_csv(
    "/kaggle/input/competitions/heavy-equipment-selling-price-prediction-challenge/sample_submission.csv"
)

/tmp/ipykernel_16/1619439496.py:1: DtypeWarning: Columns (38,39) have mixed types. Specify dtype option on import or set low_memory=False.
  train = pd.read_csv(


In [3]:
y = train["TargetValue"]

X = train.drop("TargetValue", axis=1)

print(X.shape)
print(y.shape)

(138701, 49)
(138701,)


In [4]:
from sklearn.dummy import DummyRegressor
dummy = DummyRegressor(strategy="mean")
dummy.fit(X,y)

DummyRegressor()

In [5]:
print(test.shape)
test.columns

(15000, 49)


Index(['TransactionID', 'AssetID', 'ProductConfigID', 'DataOriginCode',
       'VendorPartnerID', 'ManufactureYear', 'OperationalHoursMeter',
       'UtilizationTier', 'TransactionDate', 'Spec_FullDescriptor',
       'Spec_BaseClass', 'Spec_SubClass', 'Spec_ReleaseSeries',
       'Spec_VariantModifier', 'AssetScaleFactor', 'FunctionalClassification',
       'RegionCode', 'InventoryGroupCategory', 'InventoryGroupDescription',
       'col1', 'CabinType', 'Forks', 'col3', 'col4', 'DrivetrainType', 'col5',
       'col6', 'col7', 'col8', 'col9', 'col10', 'col11', 'col12', 'col13',
       'col14', 'col15', 'col16', 'col18', 'col19', 'col20', 'col21', 'col22',
       'col23', 'col24', 'col25', 'col27', 'col28', 'col29', 'col30'],
      dtype='object')

In [6]:
predictions = dummy.predict(test)

print(predictions)

print(predictions[:10])

predictions.shape

[41521.87404705 41521.87404705 41521.87404705 ... 41521.87404705
 41521.87404705 41521.87404705]
[41521.87404705 41521.87404705 41521.87404705 41521.87404705
 41521.87404705 41521.87404705 41521.87404705 41521.87404705
 41521.87404705 41521.87404705]


(15000,)

In [7]:
submission["TargetValue"] = predictions

In [8]:
submission.head()

,TransactionID,TargetValue
0,1139307,41521.874047
1,1139419,41521.874047
2,1139482,41521.874047
3,1139522,41521.874047
4,1139684,41521.874047


In [9]:
submission["TargetValue"] = predictions

In [10]:
submission.head()

,TransactionID,TargetValue
0,1139307,41521.874047
1,1139419,41521.874047
2,1139482,41521.874047
3,1139522,41521.874047
4,1139684,41521.874047


In [11]:
submission.to_csv("submission.csv", index=False)

# MILESTONE 2 Questions

In [12]:
#Creating a working copy of training data to answer MS 2 Questions

train_fe = train.copy()

train_fe["TransactionDate"].head()

0    2005-03-26
1    2009-12-18
2    2005-08-26
3    2006-11-17
4    2010-08-27
Name: TransactionDate, dtype: object

In [13]:
#converting it into DATEtime

train_fe["TransactionDate"] = pd.to_datetime(train_fe["TransactionDate"])

In [14]:
train_fe["TransactionDate"].dtype

dtype('<M8[ns]')

## Feature 1 - Finding Transaction Year

In [15]:
# Feature 1 - Finding Transaction Year
train_fe["TransactionYear"] = train_fe["TransactionDate"].dt.year

# Check
train_fe[["TransactionDate", "TransactionYear"]].head()

,TransactionDate,TransactionYear
0,2005-03-26,2005
1,2009-12-18,2009
2,2005-08-26,2005
3,2006-11-17,2006
4,2010-08-27,2010


## Feature 2 -  Finding Transaction Quarter

In [16]:
# Feature 2 -  Finding Transaction Quarter
train_fe["TransactionQuarter"] = train_fe["TransactionDate"].dt.quarter

#Check
train_fe[["TransactionDate", "TransactionQuarter"]].head()

,TransactionDate,TransactionQuarter
0,2005-03-26,1
1,2009-12-18,4
2,2005-08-26,3
3,2006-11-17,4
4,2010-08-27,3


## Feature 3 - Finding asset age 

In [17]:
# Feature 3 - Finding asset age 

train_fe["AssetAge"] = train_fe['TransactionYear'] - train_fe["ManufactureYear"]

#Check

train_fe[["ManufactureYear", "TransactionYear", "AssetAge"]].head()

,ManufactureYear,TransactionYear,AssetAge
0,1997,2005,8
1,2005,2009,4
2,1994,2005,11
3,2002,2006,4
4,2009,2010,1


## Milestone 2 Question 1

Which AssetAge occurs most frequently in the training dataset?

In [18]:
# Solution to Q1
train_fe["AssetAge"].mode()

0    5
Name: AssetAge, dtype: int64

## Milestone 2 Question 2

Which  TransactionQuarter has the highest average TargetValue?

In [19]:
# Solution to Q2
train_fe.groupby("TransactionQuarter")["TargetValue"].mean()

TransactionQuarter
1    42873.587453
2    41408.258228
3    40619.200589
4    40562.454024
Name: TargetValue, dtype: float64